# Customer Churn Prediction - Exploratory Data Analysis
In this project, I aim to predict which telecom customers are likely to cancel their subscription(churn). By identifying at-risks customers, the business can intervene with retention offers, saving revenue.

In [ ]:
import numpy as  np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style = "whitegrid")

# Load the data
df = pd.read_csv("../data/telco_customer_churn.csv")

# Inspect data
display(df.head())
df.info()

In [ ]:
# Check for the missing values
print(df.isna().sum())

# Notice 'TotalCharges' might be an object type instead of float. Check it:
print(df["TotalCharges"].dtype)
# If it's an object, it might contain blank spaces instead of NaNs.
# Let's see unique values that aren't numbers.
print(df["TotalCharges"].unique()[:10])

In [ ]:
# Replace spaces(" ") into np.nan and convert column to float.
df["TotalCharges"] = df["TotalCharges"].replace(" ", np.nan)
df["TotalCharges"] = df["TotalCharges"].astype(float)

print(df["TotalCharges"].dtype)
print(df["TotalCharges"].isna().sum())

In [ ]:
# Bar plot for the distribution of churn
fig,ax = plt.subplots(figsize = (6,4))
sns.countplot(data = df, x = "Churn", hue = "Churn", palette = "Set1", ax = ax )
ax.set_title("Distribution of Churn")
ax.legend(title = "Churn", labels = ["No","Yes"])

In [ ]:
# Percentage of customer churned vs stayed
print(df["Churn"].value_counts(normalize=True)*100)

The data is imbalance. Approximately 73% of the customers didnot churn, while 26% churned. Since the non churn class is much greater than churn class, the data is not evenly distributed.

## Data preprocessing
Machine learning model requires numerical data. I will drop unnecessary columns, handle the missing values, encode the target variable, and one-hot encode the categorial features. 

In [ ]:
# 1. Drop the customerID because (it's unique for every person and has no predictive power.)
df = df.drop(columns = ['customerID'])

# 2. Drop the 11 rows with missing TotalCharges (it's less than 0.2% of the data, so safe to drop)
df = df.dropna(subset = ['TotalCharges'])

# 3. Convert the target variable Churn to 1(Yes) and 0(No)
df['Churn'] = df['Churn'].map({'Yes':1, 'No':0})

# 4. Convert categorical variables into dummy/indicator variables (One-Hot Encoding)
# We drop the first column to avoid the "dummy variable trap" (multicollinearity)
df_encoded = pd.get_dummies(df, drop_first=True)

print(f"Original Shape: {df.shape}")
print(f"Encoded Shape: {df_encoded.shape}")
display(df_encoded.head())

Now that the data is fully numerical, I will separate it into Features(X) and Target(y) and split it into Training and Testing sets.

In [ ]:
from sklearn.model_selection import train_test_split

# Separate Features and Target
X = df_encoded.drop(columns = ['Churn'])
y = df_encoded["Churn"]

# Train/Test split(80% train and 20% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify = y)

print(f"Training Set: {X_train.shape}")
print(f"Testing Set: {X_test.shape}")

The number of columns increased because One-Hot Encoding converted categorical columns into multiple binary (0/1) columns. For example, a single column such as InternetService was split into separate columns like InternetService_Fiber optic and InternetService_No, allowing machine learning models to work with categorical data in numeric form.

## Baseline Modeling: Logistic Regression
I will train a Logistic Regression model, because dataset is imbalanced(73% No, 27% Yes), I will use "class_weight='balanced'" so the model doesn't just ignore the minority class. I will evaluate the model using Precision, Recall, and the Confusion Matrix.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# 1. Initialize the model with class_weight='balanced'
log_reg = LogisticRegression(max_iter=1000, class_weight = 'balanced', random_state=42)

# 2. Train the model
log_reg.fit(X_train,y_train)

# 3. Make the prediction
y_pred = log_reg.predict(X_test)

# 4. Evaluate
print(f"Accuracy: {accuracy_score(y_test,y_pred)*100:.2f}%")
print("Classification Report:")
print(classification_report(y_test,y_pred))

print("\n Confusion matrix:")
print(confusion_matrix(y_test,y_pred))



## Feature Scaling
This model reached the iteration limit because the features like TotalCharges (thousands) and SeniorCitizen(0/1) are on vastly different scales. I will use StandardScale to normalize the training data, which helps the model to converge faster and find a better solution.

In [ ]:
from sklearn.preprocessing import StandardScaler

# 1. Initialize the scaler
scaler = StandardScaler()

# 2. Fit on the training data and transform both train and test sets
# We ONLY fit on X_train to avoid the data leakage from test set!
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 3. Re-train the Logistic Regression on SCALED data.
# (We can also drop max iter since it will converge easily, but we'll leave it for safety.)
log_reg_scaled = LogisticRegression(class_weight='balanced', random_state=42)
log_reg_scaled.fit(X_train_scaled, y_train)

# 4. Make new predictions
y_pred_scaled = log_reg_scaled.predict(X_test_scaled)

# 5. Evalute
print(f"Scaled Accuracy score: {accuracy_score(y_test, y_pred_scaled)*100:.2f}%")

print("Scaled Classification Report:")
print(classification_report(y_test, y_pred_scaled))

print("Scaled Confusion matrix: \n")
print(confusion_matrix(y_test, y_pred_scaled))

After applying StandardScaler, Logistic Regression converged successfully without warnings. Scaling mainly improved the training process and convergence speed, while the model accuracy remained approximately the same.

## Advanced Modeling: XGBoost
While Logistic Regression is good baseline model, it is linear model. I will train an XGBoost classifier, which uses gradient boost decision trees to capture complex, non-linear relationships in the data. To handle the class imbalance, I will use the 'scale_pos_weight' parameter.

In [ ]:
from xgboost import XGBClassifier
import time

# XGBoost has a parameter to handle imbalance called scale_pos_weight
# It's roughly the ratio of negative class to positive class
# 73% stayed / 27% churn ≈ 3
neg_count = (y_train == 0).sum()
pos_count = (y_train == 1).sum()
scale_weight = neg_count/pos_count

# 1. Initialize XGBoost model
xgb_model = XGBClassifier(
    n_estimators=100,
    learning_rate = 0.1,
    max_depth = 3,
    scale_pos_weight = scale_weight, # Handles the weight
    ranndom_state=42,
    eval_metric='logloss'
)

# 2. Train the model(and time it)
start_time = time.time()
xgb_model.fit(X_train, y_train) # Note: XGBoost doesn't strictly require scaling
end_time = time.time()

print(f"Training took: {end_time - start_time:.2f}seconds.\n")

# 3. Predict
y_pred_xgb = xgb_model.predict(X_test)

# 4. Evaluate
print(f"Accuracy: {accuracy_score(y_test, y_pred_xgb)*100:.2f}%")

print("Classification Report:")
print(classification_report(y_test, y_pred_xgb))

print("Confusion matrix: \n")
print(confusion_matrix(y_test, y_pred_xgb))



## Model Comparison & Conclusion
I trained and compared three models to solve the customer churn problem:

1. **Logistic Regression (Unscaled):** Served as the baseline but struggled with convergence issues due to unscaled features.
2. **Logistic Regression (Scaled):** Fixed the convergence issues, resulting in reliable, stable predictions.
3. **XGBoost:** Delivered the best overall performance. By utilizing `scale_pos_weight` to handle the class imbalance, it achieved the highest accuracy (72.99%) and successfully reduced the number of False Negatives (missed churners) to 73.

**Best Model for Business:** 
XGBoost is the winner. While its accuracy is only slightly higher than Logistic Regression, it is better at capturing complex, non-linear relationships. Catching more at-risk customers (higher Recall) allows the business to deploy targeted retention strategies, ultimately saving more revenue.

## Feature Importance
Predicting churn is only half the battle; the business needs to know *why* customers are leaving to fix the root cause. I will extract the feature importances from XGBoost model to identify the strongest drivers of churn.

In [ ]:
# Get feature importances from the trained XGBoost model
importances = xgb_model.feature_importances_

# Create dataframe for easy viewing
feature_imp_df = pd.DataFrame({
    'Feature': X.columns,
    'Importance' : importances
}).sort_values(by = 'Importance', ascending= False)

# Plot the top 15 features
fig, ax = plt.subplots(figsize = (10,6))
sns.barplot(data = feature_imp_df.head(15), x = 'Importance', y = 'Feature', hue='Feature', palette='viridis', ax=ax)
ax.set_title('Top 15 Feature Driving Customer Churn (XGBoost)')
ax.set_xlabel('Relative Importance')
ax.set_ylabel('Feature')
plt.tight_layout()

plt.show()

## Feature Importance & Business Insights
Based on the XGBoost model, the top features driving customer churn are:
1. **Contract Type (1-year and 2-year):** Because "Month-to-month" was the baseline dropped during encoding, these features indicate that long-term contracts strongly prevent churn. 
2. **Internet Service (Fiber Optic):** Fiber optic customers are highly likely to churn, suggesting potential dissatisfaction with pricing or service quality in that segment.
3. **Tenure:** How long a customer has stayed is a major predictor of whether they will leave.

**Business Advice for the CEO:**
- **Incentivize Annual Contracts:** The company should heavily push or discount 1-year and 2-year contracts to lock customers in and prevent them from leaving.
- **Investigate Fiber Optic:** We need to survey Fiber Optic users to find out why they are leaving. Is our pricing too high compared to competitors? Are there technical issues?
- **Target New Customers:** Since `tenure` is highly important, the first 6-12 months are critical. We should offer extra perks to new customers to get them past the initial churn risk window.